# Credit Card Fraud Detection — Model Training

**Dataset:** [Kaggle — Credit Card Fraud Detection (mlg-ulb)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
**Environment:** Kaggle Notebook (training) → artifacts exported for local FastAPI + Streamlit app

This notebook covers:
1. Data exploration and inspection
2. Exploratory Data Analysis (EDA)
3. Handling extreme class imbalance
4. Feature scaling
5. Train/test split
6. Training and comparing Logistic Regression, Random Forest, XGBoost (+ Isolation Forest for anomaly-detection comparison)
7. Evaluation with fraud-relevant metrics
8. Model selection and justification
9. Serialization of the final model + scaler for deployment

> **Note on the dataset:** `creditcard.csv` has 284,807 transactions, of which only 492 (0.172%) are fraud — this is one of the most imbalanced classification problems commonly used for teaching. Never trust accuracy alone here: a model that predicts "legitimate" for everything would score 99.8% accuracy and catch zero fraud.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, precision_recall_curve
)
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from xgboost import XGBClassifier

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
RANDOM_STATE = 42

## Phase 1 — Data Exploration and Preprocessing

### Step 1: Load and Inspect Data

In [ ]:
# On Kaggle, the dataset is typically mounted at:
# /kaggle/input/creditcardfraud/creditcard.csv
# Adjust the path below if running locally.
DATA_PATH = "/kaggle/input/creditcardfraud/creditcard.csv"

try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    # Fallback for local runs — place creditcard.csv next to this notebook
    df = pd.read_csv("creditcard.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum().sum(), "total missing values")
df.describe()

### Step 2: Exploratory Data Analysis

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print("Class counts:\n", class_counts)
print("\nClass percentages:\n", class_pct.round(4))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Class', data=df, ax=ax[0])
ax[0].set_title("Class Distribution (counts)")
ax[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

ax[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
          colors=['#2a9d8f', '#e76f51'])
ax[1].set_title("Class Distribution (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 12))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.1)
plt.title("Correlation Heatmap — All Features")
plt.show()

In [ ]:
# Distribution of Amount and Time
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Amount'], bins=50, ax=ax[0], color='#2a9d8f')
ax[0].set_title("Distribution of Transaction Amount")

sns.histplot(df['Time'], bins=50, ax=ax[1], color='#264653')
ax[1].set_title("Distribution of Time (seconds since first transaction)")
plt.tight_layout()
plt.show()

In [ ]:
# Fraud vs non-fraud comparison for Amount
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Class', y='Amount', data=df, ax=ax[0])
ax[0].set_title("Amount: Fraud vs Legitimate")
ax[0].set_xticklabels(['Legitimate', 'Fraud'])

# Zoomed view (fraud amounts are usually smaller and less spread out)
sns.boxplot(x='Class', y='Amount', data=df[df['Amount'] < 500], ax=ax[1])
ax[1].set_title("Amount (< 500): Fraud vs Legitimate")
ax[1].set_xticklabels(['Legitimate', 'Fraud'])
plt.tight_layout()
plt.show()

print(df.groupby('Class')['Amount'].describe())

In [ ]:
# A few of the most correlated PCA components vs Class, for context
top_corr_features = corr['Class'].abs().sort_values(ascending=False).index[1:6]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, feat in enumerate(top_corr_features):
    sns.boxplot(x='Class', y=feat, data=df, ax=axes[i])
    axes[i].set_xticklabels(['Legit', 'Fraud'])
plt.tight_layout()
plt.show()

### Step 3: Address Class Imbalance

We compare four strategies. Each has trade-offs:

| Technique | How it works | Pros | Cons |
|---|---|---|---|
| **Random Undersampling** | Randomly removes majority-class rows until classes are balanced | Fast, simple, no synthetic data | Throws away a huge amount of legitimate-transaction information → higher variance, risk of underfitting |
| **Random Oversampling** | Duplicates minority-class rows | Keeps all majority data | Duplicate rows can cause overfitting to the exact fraud examples seen |
| **SMOTE** | Generates synthetic minority samples by interpolating between existing fraud cases and their nearest neighbors | Adds diversity vs. plain duplication, keeps majority data | Synthetic points can sit in unrealistic regions of feature space; slower on huge datasets |
| **Class-weight adjustment** | Leaves data untouched, penalizes the model more for misclassifying the minority class | No data manipulation, no risk of synthetic artifacts | Only works for models that support `class_weight`; doesn't help models that don't (e.g. plain boosting without `scale_pos_weight`) |

We compare **class-weight adjustment**, **SMOTE**, and demonstrate undersampling, applying the imbalance-handling *only after* the train/test split so the test set stays a realistic, untouched sample of real-world fraud rates.

### Step 4: Feature Scaling

In [ ]:
# Only Time and Amount need scaling — V1-V28 are already PCA-transformed (mean ~0, unit-ish variance)
scaler = StandardScaler()
df[['Time', 'Amount']] = scaler.fit_transform(df[['Time', 'Amount']])
df[['Time', 'Amount']].describe()

### Step 5: Train-Test Split (stratified)

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, "| Fraud rate:", y_train.mean().round(5))
print("Test shape:", X_test.shape, "| Fraud rate:", y_test.mean().round(5))

In [ ]:
# Undersampling and SMOTE applied to the TRAINING set only (test set stays untouched/realistic)
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)
print("Undersampled training set:", y_train_rus.value_counts().to_dict())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("SMOTE training set:", y_train_sm.value_counts().to_dict())

## Phase 2 — Model Training and Selection

We train three required models plus Isolation Forest as an optional anomaly-detection baseline:

1. **Logistic Regression** — baseline, `class_weight='balanced'`, trained on the original (unresampled) training set
2. **Random Forest** — trained on the SMOTE-resampled training set, nonlinear relationships, feature importance
3. **XGBoost** — trained on the original training set using `scale_pos_weight` (XGBoost's native equivalent of class-weighting) for imbalance
4. **Isolation Forest** (optional) — unsupervised anomaly detector, trained without labels, included for comparison against the supervised models

In [ ]:
results = {}

# Model 1: Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
results['Logistic Regression'] = lr

# Model 2: Random Forest (trained on SMOTE-balanced data)
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)
results['Random Forest'] = rf

# Model 3: XGBoost
scale_pos_weight = (y_train.value_counts()[0] / y_train.value_counts()[1])
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight, eval_metric='logloss',
    random_state=RANDOM_STATE
)
xgb.fit(X_train, y_train)
results['XGBoost'] = xgb

print("All models trained.")

In [ ]:
# Random Forest feature importance
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
importances.head(15).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title("Random Forest — Top 15 Feature Importances")
plt.show()

In [ ]:
# Optional: Isolation Forest (unsupervised anomaly detection, no labels used during fit)
iso_forest = IsolationForest(
    n_estimators=200, contamination=y_train.mean(), random_state=RANDOM_STATE
)
iso_forest.fit(X_train)

# IsolationForest outputs -1 (anomaly) / 1 (normal); map to 1 = fraud, 0 = legit
iso_pred_test = np.where(iso_forest.predict(X_test) == -1, 1, 0)
print("Isolation Forest — treated as anomaly detector (for comparison only, not part of the model shortlist):")
print(classification_report(y_test, iso_pred_test, target_names=['Legit', 'Fraud']))

### Model Evaluation

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        'Model': name,
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1 Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'PR-AUC': average_precision_score(y_test, proba),
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn
    }
    return metrics, cm, proba

comparison_rows = []
roc_data = {}
for name, model in results.items():
    metrics, cm, proba = evaluate_model(name, model, X_test, y_test)
    comparison_rows.append(metrics)
    roc_data[name] = proba

    plt.figure(figsize=(4, 3.5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    plt.title(f"Confusion Matrix — {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
comparison_df.round(4)

In [ ]:
# Precision-Recall curves — the metric that matters most under extreme imbalance
plt.figure(figsize=(8, 6))
for name, proba in roc_data.items():
    precision, recall, _ = precision_recall_curve(y_test, proba)
    plt.plot(recall, precision, label=name)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.show()

### Model Comparison and Selection

Review `comparison_df` above. In fraud detection:

- **Recall** matters most — a missed fraud (false negative) usually costs far more than a false alarm on a legitimate transaction (false positive), which just means a customer gets a verification prompt.
- **Precision** still matters — too many false positives creates alert fatigue for the fraud team and annoys legitimate customers.
- **PR-AUC** (not ROC-AUC) is the more honest metric here, since ROC-AUC can look deceptively good on highly imbalanced data — PR-AUC focuses specifically on how well the model finds the rare positive class.

**Selection rule used below:** rank by PR-AUC first (most reliable summary metric for imbalanced fraud data), then break ties by Recall, since catching fraud is the priority business objective.

In [ ]:
best_model_name = comparison_df.sort_values(['PR-AUC', 'Recall'], ascending=False).index[0]
best_model = results[best_model_name]
print(f"Selected model: {best_model_name}")
print(comparison_df.loc[[best_model_name]])

**Justification:** the selected model is documented in `reports/technical_report.md` and should be filled in with the actual numbers once this notebook is run against the real `creditcard.csv` on Kaggle. As a rule of thumb on this dataset, tree-based models (Random Forest / XGBoost) typically edge out plain Logistic Regression on PR-AUC and Recall because fraud patterns are non-linear, but Logistic Regression remains a fast, interpretable baseline worth reporting alongside it.

## Model Serialization

Save the selected model and the fitted scaler so the FastAPI backend can load them directly.

In [ ]:
from xgboost import XGBClassifier

joblib.dump(scaler, "scaler.pkl")

if isinstance(best_model, XGBClassifier):
    # Native XGBoost format — stable across XGBoost versions, avoids the
    # "loading a serialized model from an older version" pickle warning
    # that can show up when training and serving environments differ.
    best_model.save_model("fraud_model.json")
    print("Saved fraud_model.json (XGBoost native format) and scaler.pkl")
else:
    # Logistic Regression / Random Forest — plain joblib is fine, no
    # cross-version pickle concerns for these.
    joblib.dump(best_model, "fraud_model.pkl")
    print("Saved fraud_model.pkl and scaler.pkl")

print("Copy the model file(s) above into the backend/ folder before starting the API.")

In [ ]:
# Sanity check: reload and predict on one held-out test row
import os

reloaded_scaler = joblib.load("scaler.pkl")

if os.path.exists("fraud_model.json"):
    reloaded_model = XGBClassifier()
    reloaded_model.load_model("fraud_model.json")
else:
    reloaded_model = joblib.load("fraud_model.pkl")

sample = X_test.iloc[[0]]
pred = reloaded_model.predict(sample)[0]
proba = reloaded_model.predict_proba(sample)[0][1]
print(f"Reloaded model sanity check -> prediction: {pred}, fraud probability: {proba:.4f}")

## Summary

- Loaded and inspected the dataset, confirmed extreme class imbalance (~0.17% fraud).
- Performed EDA: class distribution, correlation heatmap, Amount/Time distributions, fraud vs. legitimate boxplots.
- Compared undersampling, SMOTE, and class-weighting to handle imbalance.
- Scaled `Time` and `Amount` with `StandardScaler`; saved the fitted scaler for reuse at inference time.
- Trained and compared Logistic Regression, Random Forest, and XGBoost, plus an Isolation Forest baseline.
- Evaluated with Precision, Recall, F1, ROC-AUC, and PR-AUC — prioritizing Recall/PR-AUC given the cost asymmetry of missed fraud.
- Selected the best model and serialized it (`fraud_model.pkl`) alongside the scaler (`scaler.pkl`) for the FastAPI backend.

**Next step:** move `fraud_training.ipynb`'s output files (`fraud_model.pkl`, `scaler.pkl`) into `backend/`, then follow the README to run the FastAPI backend and Streamlit frontend.